In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import h5py
from datetime import datetime
from scipy.signal import savgol_filter
from scipy.ndimage import gaussian_filter1d, zoom
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

FILEPATH = "/kaggle/input/datasets/yvsuyashmurthy/kerala-coast-mineral-deposits/PRS_L1_STD_OFFL_20220110052918_20220110052923_0001.he5"
USGS_DIR = "/kaggle/input/datasets/yvsuyashmurthy/usgs-spectra"

# Processing parameters
SCALE_FACTOR = 10000.0
SMOOTH_WINDOW = 11
SMOOTH_POLY = 3
GAUSSIAN_SIGMA = 1.2

# Water masking
WATER_SWIR_MIN_WL = 1.6
WATER_MEAN_THRESH = 0.025

# Destriping
ROW_BAD_MAD_THRESHOLD = 4.0

# Band depth features
BAND_FEATURES = {
    "Al_OH_2200": {"center": 2.20, "left": 0.06, "right": 0.06},
    "Carbonate_2330": {"center": 2.33, "left": 0.04, "right": 0.04},
    "MgFe_2350": {"center": 2.35, "left": 0.05, "right": 0.05},
    "OH_1900": {"center": 1.90, "left": 0.03, "right": 0.03},
}

# USGS library
USGS_FILES = {
    "ilmenite": "Ilmenite.txt",
    "rutile": "Rutile.txt",
    "zircon": "Zircon.txt",
    "garnet": "Andradite.txt",
    "calcite": "Calcite.txt",
    "dolomite": "Dolomite.txt",
    "albite": "Albite.txt",
    "kaolinite": "Kaolinite.txt",
}

# SAM thresholds (degrees) - MUCH MORE RELAXED based on actual data
SAM_THRESH = {
    "ilmenite": 50,    # Observed: 40-64°
    "rutile": 60,      # Observed: 56-80°
    "zircon": 55,      # Observed: 50-73°
    "garnet": 52,      # Observed: 44-66°
    "calcite": 48,     # Observed: 39-61°
    "dolomite": 50,    # Observed: 39-62°
    "albite": 50,      # Observed: 40-63°
    "kaolinite": 50,   # Observed: 46-65°
}

# Classification threshold
MIN_CONFIDENCE = 0.15  # Lowered for better detection

def load_prisma_data(filepath):
    """Load PRISMA HDF5 data with correct axis ordering."""
    print("\n" + "="*70)
    print("STEP 1: LOADING PRISMA DATA")
    print("="*70)
    
    data = {}
    with h5py.File(filepath, "r") as f:
        vnir_raw = np.array(f["HDFEOS/SWATHS/PRS_L1_HCO/Data Fields/VNIR_Cube"])
        swir_raw = np.array(f["HDFEOS/SWATHS/PRS_L1_HCO/Data Fields/SWIR_Cube"])
        
        print(f"Raw VNIR shape: {vnir_raw.shape}")
        print(f"Raw SWIR shape: {swir_raw.shape}")
        
        data["vnir_cube"] = np.transpose(vnir_raw, (0, 2, 1))
        data["swir_cube"] = np.transpose(swir_raw, (0, 2, 1))
        
        print(f"Transposed VNIR: {data['vnir_cube'].shape}")
        print(f"Transposed SWIR: {data['swir_cube'].shape}")
        
        data["cloud_mask"] = np.array(
            f["HDFEOS/SWATHS/PRS_L1_HCO/Data Fields/Cloud_Mask"]
        ) > 0
        data["sunglint_mask"] = np.array(
            f["HDFEOS/SWATHS/PRS_L1_HCO/Data Fields/SunGlint_Mask"]
        ) > 0
        
        try:
            wv_vnir = np.array(f["HDFEOS/SWATHS/PRS_L1_HCO/Ancillary/Wavelength_VNIR"])
            wv_swir = np.array(f["HDFEOS/SWATHS/PRS_L1_HCO/Ancillary/Wavelength_SWIR"])
            data["wavelengths_vnir"] = wv_vnir / 1000 if wv_vnir.max() > 100 else wv_vnir
            data["wavelengths_swir"] = wv_swir / 1000 if wv_swir.max() > 100 else wv_swir
            print(f"Wavelengths: VNIR {data['wavelengths_vnir'].min():.3f}-{data['wavelengths_vnir'].max():.3f} μm")
        except:
            print("Using default wavelengths")
            data["wavelengths_vnir"] = np.linspace(0.4, 1.0, 66)
            data["wavelengths_swir"] = np.linspace(1.0, 2.5, 173)
    
    return data

def atmospheric_correction_toa(data, scale=SCALE_FACTOR):
    """Simple DN to reflectance scaling."""
    print("\n" + "="*70)
    print("STEP 2: ATMOSPHERIC CORRECTION")
    print("="*70)
    
    out = data.copy()
    vnir = np.clip(data["vnir_cube"].astype(np.float32) / scale, 0, 1)
    swir = np.clip(data["swir_cube"].astype(np.float32) / scale, 0, 1)
    
    mask = data["cloud_mask"] | data["sunglint_mask"]
    vnir[mask] = np.nan
    swir[mask] = np.nan
    
    print(f"VNIR reflectance: [{vnir[~np.isnan(vnir)].min():.4f}, {vnir[~np.isnan(vnir)].max():.4f}]")
    print(f"SWIR reflectance: [{swir[~np.isnan(swir)].min():.4f}, {swir[~np.isnan(swir)].max():.4f}]")
    print(f"Masked pixels: {mask.sum():,} ({100*mask.mean():.1f}%)")
    
    out["reflectance_vnir"] = vnir
    out["reflectance_swir"] = swir
    return out

def get_rgb_quicklook(vnir_refl, wavelengths_um, stretch_pct=(2, 98), gamma=0.45):
    """Create RGB quicklook."""
    idx_r = int(np.argmin(np.abs(wavelengths_um - 0.650)))
    idx_g = int(np.argmin(np.abs(wavelengths_um - 0.560)))
    idx_b = int(np.argmin(np.abs(wavelengths_um - 0.470)))
    
    rgb = np.stack([vnir_refl[:, :, idx_r], 
                    vnir_refl[:, :, idx_g], 
                    vnir_refl[:, :, idx_b]], axis=-1)
    rgb = np.nan_to_num(rgb, nan=0.0)
    
    for i in range(3):
        channel = rgb[:, :, i]
        valid = (channel > 0) & (channel < 1)
        if np.any(valid):
            p_low = np.percentile(channel[valid], stretch_pct[0])
            p_high = np.percentile(channel[valid], stretch_pct[1])
            rgb[:, :, i] = (channel - p_low) / (p_high - p_low + 1e-9)
    
    rgb = np.clip(rgb, 0, 1) ** gamma
    return rgb

def combine_vnir_swir(vnir, swir, wv_vnir, wv_swir):
    """Combine VNIR and SWIR with atmospheric window removal."""
    print("\n" + "="*70)
    print("STEP 4: COMBINING VNIR + SWIR")
    print("="*70)
    
    if vnir.shape[:2] != swir.shape[:2]:
        print(f"Resampling SWIR: {swir.shape} → ({vnir.shape[0]}, {vnir.shape[1]}, {swir.shape[2]})")
        swir = zoom(swir, (vnir.shape[0]/swir.shape[0], vnir.shape[1]/swir.shape[1], 1), order=1)
    
    good_v = (wv_vnir >= 0.42) & ~((wv_vnir > 0.93) & (wv_vnir < 1.0))
    good_s = ~((wv_swir > 1.35) & (wv_swir < 1.45)) & \
             ~((wv_swir > 1.8) & (wv_swir < 1.95)) & (wv_swir < 2.47)
    
    print(f"Good bands: VNIR {good_v.sum()}/{len(wv_vnir)}, SWIR {good_s.sum()}/{len(wv_swir)}")
    
    cube = np.concatenate([vnir[:, :, good_v], swir[:, :, good_s]], axis=2)
    wl = np.concatenate([wv_vnir[good_v], wv_swir[good_s]])
    
    valid = ~np.isnan(cube).any(axis=2)
    valid &= np.nanmean(cube, axis=2) > 0.002
    
    print(f"Full cube: {cube.shape}")
    print(f"Valid pixels: {valid.sum():,} / {valid.size:,} ({100*valid.mean():.1f}%)")
    
    return cube.astype(np.float32), wl.astype(float), valid

def mask_water(full_cube, full_wl, valid_mask):
    """Detect water pixels using SWIR reflectance."""
    print("\n" + "="*70)
    print("STEP 5: WATER DETECTION")
    print("="*70)
    
    swir_idx = np.where(full_wl >= WATER_SWIR_MIN_WL)[0]
    if swir_idx.size == 0:
        water_mask = np.zeros(full_cube.shape[:2], dtype=bool)
    else:
        swir_mean = np.nanmean(full_cube[:, :, swir_idx], axis=2)
        water_mask = swir_mean < WATER_MEAN_THRESH
    
    print(f"Water pixels detected: {water_mask.sum():,} / {water_mask.size:,} ({100*water_mask.mean():.1f}%)")
    print("Water mask will be applied to visualizations and classification")
    
    return water_mask

def destripe_rows(full_cube, valid_mask, threshold_mad=ROW_BAD_MAD_THRESHOLD):
    """Remove row striping artifacts."""
    print("\n" + "="*70)
    print("STEP 6: DESTRIPING")
    print("="*70)
    
    rows, cols, bands = full_cube.shape
    row_means = np.nanmean(full_cube.reshape(rows, -1), axis=1)
    
    med = np.nanmedian(row_means)
    mad = np.nanmedian(np.abs(row_means - med))
    
    if mad <= 0:
        print("No striping detected")
        return full_cube
    
    bad = (np.isnan(row_means)) | (np.abs(row_means - med) > threshold_mad * mad)
    
    if not np.any(bad):
        print("No bad rows detected")
        return full_cube
    
    print(f"Bad rows detected: {bad.sum()}")
    
    good_idx = np.where(~bad)[0]
    full_out = full_cube.copy()
    
    for r in np.where(bad)[0]:
        above = good_idx[good_idx < r]
        below = good_idx[good_idx > r]
        
        if above.size == 0 and below.size == 0:
            continue
        elif above.size == 0:
            full_out[r] = full_cube[below[0]]
        elif below.size == 0:
            full_out[r] = full_cube[above[-1]]
        else:
            a, b = above[-1], below[0]
            weight = (r - a) / (b - a)
            full_out[r] = (1 - weight) * full_cube[a] + weight * full_cube[b]
    
    return full_out

def smooth_spectra(full_cube, window=SMOOTH_WINDOW, poly=SMOOTH_POLY):
    """Smooth spectra using Savitzky-Golay filter."""
    print("\n" + "="*70)
    print("STEP 7: SPECTRAL SMOOTHING")
    print("="*70)
    
    try:
        smoothed = savgol_filter(full_cube, window_length=window, 
                                polyorder=poly, axis=2, mode='mirror')
        
        # CRITICAL FIX: Clip negative values that result from smoothing
        smoothed = np.clip(smoothed, 0, None)
        
        print(f"Savitzky-Golay smoothing applied (window={window}, poly={poly})")
        print(f"Post-smoothing range: [{np.nanmin(smoothed):.4f}, {np.nanmax(smoothed):.4f}]")
        return smoothed.astype(np.float32)
    except Exception as e:
        print(f"Savitzky-Golay failed: {e}")
        print("Using Gaussian smoothing instead")
        smoothed = gaussian_filter1d(full_cube, sigma=GAUSSIAN_SIGMA, 
                                     axis=2, mode='reflect')
        smoothed = np.clip(smoothed, 0, None)
        return smoothed.astype(np.float32)

def compute_band_depth(wl, spectrum, center, left_offset, right_offset):
    """Calculate band depth at a specific wavelength."""
    center_idx = int(np.argmin(np.abs(wl - center)))
    R_center = spectrum[center_idx]
    
    if not np.isfinite(R_center) or R_center <= 0:
        return 0.0
    
    left_idx = int(np.argmin(np.abs(wl - (center - left_offset))))
    right_idx = int(np.argmin(np.abs(wl - (center + right_offset))))
    
    R_continuum = (spectrum[left_idx] + spectrum[right_idx]) / 2.0
    
    if not np.isfinite(R_continuum) or R_continuum <= 1e-6:
        return 0.0
    
    bd = 1.0 - (R_center / R_continuum)
    return float(np.clip(bd, 0, 1))

def compute_band_depth_maps(full_cube, full_wl, valid_mask):
    """Compute band depth maps for all features."""
    print("\n" + "="*70)
    print("STEP 8: BAND DEPTH CALCULATION")
    print("="*70)
    
    rows, cols, bands = full_cube.shape
    depth_maps = {}
    
    for name, params in BAND_FEATURES.items():
        print(f"Computing {name} @ {params['center']:.2f} μm")
        dm = np.zeros((rows, cols), dtype=np.float32)
        
        for i in range(rows):
            for j in range(cols):
                if valid_mask[i, j]:
                    dm[i, j] = compute_band_depth(
                        full_wl, full_cube[i, j], 
                        params['center'], params['left'], params['right']
                    )
        
        valid_depths = dm[valid_mask & (dm > 0)]
        if valid_depths.size > 0:
            print(f"  Range: [{valid_depths.min():.4f}, {valid_depths.max():.4f}], "
                  f"Mean: {valid_depths.mean():.4f}, "
                  f"Pixels > 0: {(dm > 0).sum():,}")
        
        depth_maps[name] = dm
    
    return depth_maps

def load_usgs_spectrum(filepath):
    """Load USGS spectral library file."""
    vals = []
    with open(filepath, 'r', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if not line or line.lower().startswith(('splib', 'record', '#')):
                continue
            try:
                vals.append(float(line.split()[0]))
            except:
                continue
    
    vals = np.array(vals)
    wl = np.linspace(0.35, 2.50, len(vals))
    return wl, vals

def load_usgs_library(usgs_dir, usgs_files, target_wl):
    """Load and resample USGS reference spectra - FIXED VERSION."""
    print("\n" + "="*70)
    print("STEP 9: LOADING USGS SPECTRAL LIBRARY")
    print("="*70)
    
    refs = {}
    for mineral, filename in usgs_files.items():
        filepath = os.path.join(usgs_dir, filename)
        if not os.path.exists(filepath):
            print(f"Warning: {filename} not found, skipping {mineral}")
            continue
        
        wl, spec = load_usgs_spectrum(filepath)
        
        # CRITICAL FIX 1: Remove extreme outliers
        spec = np.clip(spec, -10, 10)  # Remove absurd values
        
        # CRITICAL FIX 2: Normalize USGS spectrum
        spec_valid = spec[np.isfinite(spec)]
        if len(spec_valid) > 0:
            spec = (spec - np.min(spec_valid)) / (np.max(spec_valid) - np.min(spec_valid) + 1e-9)
        
        # Resample to PRISMA wavelengths
        resampled = np.interp(target_wl, wl, spec, left=np.nan, right=np.nan)
        
        # Additional normalization after resampling
        valid_idx = np.isfinite(resampled)
        if valid_idx.sum() > 10:
            resampled[valid_idx] = (resampled[valid_idx] - np.min(resampled[valid_idx])) / \
                                   (np.max(resampled[valid_idx]) - np.min(resampled[valid_idx]) + 1e-9)
        
        refs[mineral] = resampled
        print(f"  Loaded: {mineral} (valid bands: {valid_idx.sum()}, "
              f"range: [{np.nanmin(resampled):.4f}, {np.nanmax(resampled):.4f}])")
    
    print(f"Total minerals: {len(refs)}")
    return refs

def spectral_angle_mapper(spec_a, spec_b):
    """Calculate spectral angle - IMPROVED VERSION."""
    # Find valid bands in both spectra
    valid = np.isfinite(spec_a) & np.isfinite(spec_b) & (spec_a >= 0) & (spec_b >= 0)
    
    if valid.sum() < 20:  # Need at least 20 valid bands
        return np.nan
    
    a = spec_a[valid]
    b = spec_b[valid]
    
    # CRITICAL FIX: Normalize to 0-1 range first (min-max normalization)
    a_min, a_max = np.min(a), np.max(a)
    b_min, b_max = np.min(b), np.max(b)
    
    if a_max - a_min > 1e-9:
        a = (a - a_min) / (a_max - a_min)
    if b_max - b_min > 1e-9:
        b = (b - b_min) / (b_max - b_min)
    
    # Then normalize by vector length
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    
    if norm_a < 1e-9 or norm_b < 1e-9:
        return np.nan
    
    a = a / norm_a
    b = b / norm_b
    
    # Calculate angle
    dot_product = np.dot(a, b)
    cos_angle = np.clip(dot_product, -1, 1)
    angle_rad = np.arccos(cos_angle)
    
    return angle_rad

def compute_sam_scores(smoothed_cube, ref_specs, valid_mask, sam_thresholds):
    """Compute SAM scores - FIXED VERSION with diagnostics."""
    print("\n" + "="*70)
    print("STEP 10: SPECTRAL ANGLE MAPPER (SAM)")
    print("="*70)
    
    rows, cols, bands = smoothed_cube.shape
    minerals = list(ref_specs.keys())
    scores = {}
    
    # Sample pixel for diagnostics
    sample_i, sample_j = rows // 2, cols // 2
    while not valid_mask[sample_i, sample_j]:
        sample_i += 1
        sample_j += 1
        if sample_i >= rows or sample_j >= cols:
            sample_i, sample_j = rows // 2, cols // 2
            break
    
    print(f"Diagnostic: Using sample pixel at ({sample_i}, {sample_j})")
    sample_spec = smoothed_cube[sample_i, sample_j]
    print(f"  Sample spectrum range: [{np.nanmin(sample_spec):.4f}, {np.nanmax(sample_spec):.4f}]")
    print(f"  Sample spectrum mean: {np.nanmean(sample_spec):.4f}")
    
    for mineral in minerals:
        print(f"\nProcessing: {mineral}")
        score_map = np.full((rows, cols), np.nan, dtype=np.float32)
        max_angle = np.deg2rad(sam_thresholds.get(mineral, 20))
        
        # Diagnostic for reference spectrum
        ref_valid = np.isfinite(ref_specs[mineral])
        print(f"  Reference spectrum: {ref_valid.sum()} valid bands")
        print(f"  Reference range: [{np.nanmin(ref_specs[mineral]):.4f}, {np.nanmax(ref_specs[mineral]):.4f}]")
        
        # Test SAM on sample pixel
        sample_angle = spectral_angle_mapper(sample_spec, ref_specs[mineral])
        if np.isfinite(sample_angle):
            sample_sim = np.clip(1 - sample_angle / max_angle, 0, 1)
            print(f"  Sample pixel: angle={np.rad2deg(sample_angle):.2f}°, similarity={sample_sim:.4f}")
        else:
            print(f"  Sample pixel: SAM returned NaN")
        
        match_count = 0
        angle_list = []
        
        for i in range(rows):
            for j in range(cols):
                if not valid_mask[i, j]:
                    continue
                
                angle = spectral_angle_mapper(smoothed_cube[i, j], ref_specs[mineral])
                
                if np.isfinite(angle):
                    angle_list.append(angle)
                    similarity = np.clip(1 - angle / max_angle, 0, 1)
                    score_map[i, j] = similarity
                    if similarity > 0.1:
                        match_count += 1
        
        if angle_list:
            print(f"  Valid angles computed: {len(angle_list):,}")
            print(f"  Angle stats (degrees): min={np.rad2deg(np.min(angle_list)):.2f}, "
                  f"mean={np.rad2deg(np.mean(angle_list)):.2f}, "
                  f"max={np.rad2deg(np.max(angle_list)):.2f}")
            print(f"  Matches (score > 0.1): {match_count:,}")
            print(f"  Score range: [{np.nanmin(score_map):.4f}, {np.nanmax(score_map):.4f}]")
        else:
            print(f"  WARNING: No valid angles computed!")
        
        scores[mineral] = score_map
    
    return scores

def enhance_with_band_depth(scores, depth_maps):
    """Enhance mineral scores using band depth - IMPROVED VERSION."""
    print("\n" + "="*70)
    print("STEP 11: BAND DEPTH ENHANCEMENT")
    print("="*70)
    
    enhanced = scores.copy()
    
    # If SAM scores are all zero, use band depth as primary classifier
    all_zero = all(np.nanmax(s) < 0.01 for s in scores.values())
    
    if all_zero:
        print("⚠ SAM scores near zero - using BAND DEPTH as primary classifier")
        
        # Fe/Mg-bearing minerals - use MgFe depth directly
        for mineral in ['ilmenite', 'rutile', 'garnet']:
            if mineral in enhanced:
                enhanced[mineral] = depth_maps['MgFe_2350'] * 2.0  # Scale up
                print(f"  {mineral}: using MgFe_2350 depth (max={np.nanmax(enhanced[mineral]):.4f})")
        
        # Zircon - use Al-OH depth
        if 'zircon' in enhanced:
            enhanced['zircon'] = depth_maps['Al_OH_2200'] * 0.8
            print(f"  zircon: using Al_OH_2200 depth (max={np.nanmax(enhanced['zircon']):.4f})")
        
        # Carbonates - use carbonate depth
        for mineral in ['calcite', 'dolomite']:
            if mineral in enhanced:
                enhanced[mineral] = depth_maps['Carbonate_2330'] * 1.5
                print(f"  {mineral}: using Carbonate_2330 depth (max={np.nanmax(enhanced[mineral]):.4f})")
        
        # Clay minerals - use Al-OH depth
        for mineral in ['kaolinite', 'albite']:
            if mineral in enhanced:
                enhanced[mineral] = depth_maps['Al_OH_2200'] * 1.0
                print(f"  {mineral}: using Al_OH_2200 depth (max={np.nanmax(enhanced[mineral]):.4f})")
    
    else:
        # Normal enhancement (SAM + band depth)
        print("Using SAM + band depth enhancement")
        
        for mineral in ['ilmenite', 'rutile', 'garnet']:
            if mineral in enhanced:
                before = np.nanmax(enhanced[mineral])
                enhanced[mineral] *= (1 + 0.45 * depth_maps['MgFe_2350'])
                after = np.nanmax(enhanced[mineral])
                print(f"  {mineral}: {before:.4f} → {after:.4f}")
        
        if 'zircon' in enhanced:
            before = np.nanmax(enhanced['zircon'])
            enhanced['zircon'] *= (1 + 0.35 * depth_maps['Al_OH_2200'])
            after = np.nanmax(enhanced['zircon'])
            print(f"  zircon: {before:.4f} → {after:.4f}")
        
        for mineral in ['calcite', 'dolomite']:
            if mineral in enhanced:
                before = np.nanmax(enhanced[mineral])
                enhanced[mineral] *= (1 + 0.60 * depth_maps['Carbonate_2330'])
                after = np.nanmax(enhanced[mineral])
                print(f"  {mineral}: {before:.4f} → {after:.4f}")
        
        for mineral in ['kaolinite', 'albite']:
            if mineral in enhanced:
                before = np.nanmax(enhanced[mineral])
                enhanced[mineral] *= (1 + 0.50 * depth_maps['Al_OH_2200'])
                after = np.nanmax(enhanced[mineral])
                print(f"  {mineral}: {before:.4f} → {after:.4f}")
    
    return enhanced

def classify_minerals(scores, valid_mask, water_mask, min_confidence=MIN_CONFIDENCE):
    """Classify pixels to minerals - FIXED VERSION."""
    print("\n" + "="*70)
    print("STEP 12: MINERAL CLASSIFICATION")
    print("="*70)
    
    minerals = list(scores.keys())

    if not minerals:
        raise ValueError("No minerals found in scores dictionary. Cannot classify.")

    
    rows, cols = valid_mask.shape
    
    score_stack = np.stack([np.nan_to_num(scores[m], nan=0) for m in minerals], axis=2)
    
    best_idx = np.argmax(score_stack, axis=2)
    best_score = np.max(score_stack, axis=2)
    
    land_scores = best_score[valid_mask & (~water_mask)]
    if land_scores.size > 0:
        print(f"\nScore Statistics (land pixels):")
        print(f"  Min: {land_scores.min():.4f}")
        print(f"  Max: {land_scores.max():.4f}")
        print(f"  Mean: {land_scores.mean():.4f}")
        print(f"  Median: {np.median(land_scores):.4f}")
        print(f"  75th percentile: {np.percentile(land_scores, 75):.4f}")
        print(f"  95th percentile: {np.percentile(land_scores, 95):.4f}")
        
        if land_scores.max() < min_confidence:
            adaptive_thresh = max(np.percentile(land_scores, 60), 0.05)
            print(f"\n⚠ WARNING: Max score ({land_scores.max():.4f}) below threshold ({min_confidence:.4f})")
            print(f"  Using adaptive threshold: {adaptive_thresh:.4f} (60th percentile, min 0.05)")
            min_confidence = adaptive_thresh
    
    classified = np.where(
        (best_score >= min_confidence) & valid_mask & (~water_mask),
        best_idx + 1,
        0
    )
    
    land_pixels = valid_mask & (~water_mask)
    print(f"\nClassification Statistics:")
    print(f"  Threshold used: {min_confidence:.4f}")
    print(f"  Total pixels: {classified.size:,}")
    print(f"  Valid land pixels: {land_pixels.sum():,}")
    print(f"  Water pixels (masked): {water_mask.sum():,}")
    print(f"  Classified pixels: {(classified > 0).sum():,}")
    print(f"  Classification rate: {100*(classified > 0).sum()/land_pixels.sum():.1f}%")
    
    for idx, mineral in enumerate(minerals, 1):
        count = (classified == idx).sum()
        pct = 100 * count / land_pixels.sum() if land_pixels.sum() > 0 else 0
        print(f"  {mineral}: {count:,} ({pct:.2f}%)")
    
    return classified, best_score, minerals

def show_map(arr, title="", cmap='magma', figsize=(9, 7)):
    """Display a 2D map."""
    plt.figure(figsize=figsize)
    im = plt.imshow(arr, cmap=cmap)
    plt.title(title, fontsize=14, weight='bold')
    plt.colorbar(im, fraction=0.046, pad=0.04)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

def visualize_mineral_map(classified, minerals, water_mask, figsize=(12, 9)):
    """Visualize mineral classification - FIXED with water masking."""
    print("\n" + "="*70)
    print("GENERATING MINERAL MAP VISUALIZATION")
    print("="*70)
    
    # Apply water mask to classified map
    classified_display = classified.copy()
    classified_display[water_mask] = -1  # Special value for water
    
    # Create colormap
    colors = [[0, 0, 0, 1]]  # Black for unclassified
    colors.extend(plt.cm.tab20(np.linspace(0, 1, len(minerals))))
    colors.append([0.2, 0.4, 0.8, 1])  # Blue for water
    cmap = mcolors.ListedColormap(colors)
    
    plt.figure(figsize=figsize)
    plt.imshow(classified_display, cmap=cmap, interpolation='nearest',
               vmin=-1, vmax=len(minerals))
    
    cb = plt.colorbar(ticks=np.arange(-1, len(minerals) + 1), 
                     fraction=0.046, pad=0.04)
    cb.ax.set_yticklabels(['water', 'unclassified'] + minerals)
    cb.ax.tick_params(labelsize=10)
    
    plt.title('PRISMA Mineral Classification Map\n(USGS SAM + Band Depth Enhancement)', 
              fontsize=15, weight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

def create_mineral_composite(depth_maps, water_mask):
    """Create RGB composite - FIXED with water masking."""
    r = np.nan_to_num(depth_maps['Al_OH_2200'], nan=0)
    g = np.nan_to_num(depth_maps['Carbonate_2330'], nan=0)
    b = np.nan_to_num(depth_maps['MgFe_2350'], nan=0)
    
    def normalize(x):
        lo = np.nanpercentile(x, 2) if np.any(~np.isnan(x)) else 0
        hi = np.nanpercentile(x, 98) if np.any(~np.isnan(x)) else 1
        return np.clip((x - lo) / (hi - lo + 1e-9), 0, 1)
    
    rgb = np.stack([normalize(r), normalize(g), normalize(b)], axis=-1)
    
    # Mask water pixels (make them black)
    rgb[water_mask] = [0, 0, 0]
    
    plt.figure(figsize=(10, 8))
    plt.imshow(rgb)
    plt.title('Mineral Composite (water masked)\n(R: Al-OH | G: Carbonate | B: MgFe)', 
              fontsize=14, weight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

def main():
    """Execute complete PRISMA mineral mapping pipeline."""
    
    print("\n" + "="*70)
    print("PRISMA HYPERSPECTRAL MINERAL MAPPING PIPELINE")
    print("="*70)
    
    # Step 1: Load data
    data = load_prisma_data(FILEPATH)
    
    # Step 2: Atmospheric correction
    corrected = atmospheric_correction_toa(data)
    
    # Step 3: RGB quicklook
    print("\n" + "="*70)
    print("STEP 3: RGB QUICKLOOK")
    print("="*70)
    rgb = get_rgb_quicklook(corrected['reflectance_vnir'], data['wavelengths_vnir'])
    plt.figure(figsize=(10, 7))
    plt.imshow(rgb)
    plt.title('PRISMA RGB Quicklook', fontsize=14, weight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    
    # Step 4: Combine VNIR + SWIR
    full_cube, full_wl, valid_mask = combine_vnir_swir(
        corrected['reflectance_vnir'],
        corrected['reflectance_swir'],
        data['wavelengths_vnir'],
        data['wavelengths_swir']
    )
    
    # Step 5: Water detection
    water_mask = mask_water(full_cube, full_wl, valid_mask)
    
    # Step 6: Destriping
    full_cube = destripe_rows(full_cube, valid_mask)
    
    # Step 7: Spectral smoothing
    smoothed_cube = smooth_spectra(full_cube)
    
    # Step 8: Band depth maps
    depth_maps = compute_band_depth_maps(smoothed_cube, full_wl, valid_mask)
    
    # Visualize band depth maps with water masking
    for name, dm in depth_maps.items():
        dm_display = dm.copy()
        dm_display[water_mask] = np.nan
        show_map(dm_display, title=f'Band Depth: {name} ({BAND_FEATURES[name]["center"]:.2f} μm)', 
                cmap='magma')
    
    # Create mineral composite with water masking
    create_mineral_composite(depth_maps, water_mask)
    
    # Step 9: Load USGS library
    ref_specs = load_usgs_library(USGS_DIR, USGS_FILES, full_wl)
    
    # Step 10: SAM scoring
    scores = compute_sam_scores(smoothed_cube, ref_specs, valid_mask, SAM_THRESH)
    
    # Step 11: Band depth enhancement
    scores = enhance_with_band_depth(scores, depth_maps)
    
    # Step 12: Classification
    classified, confidence, minerals = classify_minerals(scores, valid_mask, water_mask)
    
    # Final visualization with water masking
    visualize_mineral_map(classified, minerals, water_mask)
    
    # Show confidence map with water masking
    confidence_display = confidence.copy()
    confidence_display[water_mask] = np.nan
    confidence_display[~valid_mask] = np.nan
    
    plt.figure(figsize=(9, 7))
    im = plt.imshow(confidence_display, cmap='viridis', vmin=0, vmax=1)
    plt.title('Classification Confidence (water masked)\n(0 = Low, 1 = High)', 
              fontsize=14, weight='bold')
    plt.colorbar(im, fraction=0.046, pad=0.04)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    
    print("\n" + "="*70)
    print("✅ PIPELINE COMPLETE!")
    print("="*70)
    print(f"Results Summary:")
    print(f"  - Full hyperspectral cube: {smoothed_cube.shape}")
    print(f"  - Wavelength range: {full_wl[0]:.3f} - {full_wl[-1]:.3f} μm")
    print(f"  - Minerals detected: {len(minerals)}")
    print(f"  - Land pixels: {(valid_mask & ~water_mask).sum():,}")
    print(f"  - Classified pixels: {(classified > 0).sum():,}")
    print(f"  - Classification rate: {100*(classified > 0).sum()/(valid_mask & ~water_mask).sum():.1f}%")
    print("="*70)
    
    results = {
        'data': data,
        'corrected': corrected,
        'full_cube': full_cube,
        'smoothed_cube': smoothed_cube,
        'full_wl': full_wl,
        'valid_mask': valid_mask,
        'water_mask': water_mask,
        'depth_maps': depth_maps,
        'ref_specs': ref_specs,
        'scores': scores,
        'classified': classified,
        'confidence': confidence,
        'minerals': minerals,
        'rgb': rgb
    }
    
    return results

if __name__ == "__main__":
    results = main()
    print("\n✅ All results stored in 'results' dictionary")
    print("   Access: results['classified'], results['depth_maps'], etc.")